In [ ]:
import os
import shutil
import sqlite3
import kagglehub
import pandas as pd

path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
print("Path dataset:", path)

raw_tables = {
    "orders": pd.read_csv(os.path.join(path, "olist_orders_dataset.csv")),
    "items": pd.read_csv(os.path.join(path, "olist_order_items_dataset.csv")),
    "payments": pd.read_csv(
        os.path.join(path, "olist_order_payments_dataset.csv")
    ),
    "customers": pd.read_csv(
        os.path.join(path, "olist_customers_dataset.csv")
    ),
    "products": pd.read_csv(os.path.join(path, "olist_products_dataset.csv")),
}

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Path dataset: /kaggle/input/brazilian-ecommerce


In [ ]:
conn = sqlite3.connect("olist_ecommerce.db")

df_orders = raw_tables["orders"].copy()
df_items = raw_tables["items"].copy()
df_payments = raw_tables["payments"].copy()
df_customers = raw_tables["customers"].copy()
df_products = raw_tables["products"].copy()

order_date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in order_date_cols:
    df_orders[col] = pd.to_datetime(df_orders[col])

df_items["shipping_limit_date"] = pd.to_datetime(
    df_items["shipping_limit_date"]
)

df_products["product_category_name"] = df_products[
    "product_category_name"
].fillna("unknown")
df_products["product_name_lenght"] = df_products[
    "product_name_lenght"
].fillna(0)
df_products["product_description_lenght"] = df_products[
    "product_description_lenght"
].fillna(0)
df_products["product_photos_qty"] = df_products["product_photos_qty"].fillna(0)
df_products["product_weight_g"] = df_products["product_weight_g"].fillna(0)
df_products["product_length_cm"] = df_products["product_length_cm"].fillna(0)
df_products["product_height_cm"] = df_products["product_height_cm"].fillna(0)
df_products["product_width_cm"] = df_products["product_width_cm"].fillna(0)

df_customers["customer_city"] = df_customers["customer_city"].str.strip()
df_customers["customer_state"] = (
    df_customers["customer_state"].str.strip().str.upper()
)

df_orders.to_sql("orders", conn, if_exists="replace", index=False)
df_items.to_sql("order_items", conn, if_exists="replace", index=False)
df_payments.to_sql("order_payments", conn, if_exists="replace", index=False)
df_customers.to_sql("customers", conn, if_exists="replace", index=False)
df_products.to_sql("products", conn, if_exists="replace", index=False)

32951

In [ ]:
for tbl in ["orders", "order_items", "order_payments", "customers", "products"]:
    cnt = pd.read_sql_query(f"SELECT COUNT(*) as total_rows FROM {tbl}", conn)[
        "total_rows"
    ][0]
    print(f"Table: {tbl:<15} | Verified Rows: {cnt}")

Table: orders          | Verified Rows: 99441
Table: order_items     | Verified Rows: 112650
Table: order_payments  | Verified Rows: 103886
Table: customers       | Verified Rows: 99441
Table: products        | Verified Rows: 32951


In [ ]:
query_delivery_performance = """
WITH delivery_metrics AS (
    SELECT
        c.customer_state,
        o.order_id,
        JULIANDAY(o.order_delivered_customer_date) - JULIANDAY(o.order_purchase_timestamp) AS actual_delivery_days,
        JULIANDAY(o.order_estimated_delivery_date) - JULIANDAY(o.order_purchase_timestamp) AS estimated_delivery_days,
        JULIANDAY(o.order_delivered_customer_date) - JULIANDAY(o.order_estimated_delivery_date) AS delay_days,
        CASE
            WHEN JULIANDAY(o.order_delivered_customer_date) <= JULIANDAY(o.order_estimated_delivery_date) THEN 1
            ELSE 0
        END AS is_on_time
    FROM
        orders o
    JOIN
        customers c ON o.customer_id = c.customer_id
    WHERE
        o.order_status = 'delivered'
        AND o.order_delivered_customer_date IS NOT NULL
        AND o.order_estimated_delivery_date IS NOT NULL
)
SELECT
    customer_state,
    COUNT(order_id) AS total_delivered_orders,
    ROUND(AVG(actual_delivery_days), 2) AS avg_actual_days,
    ROUND(AVG(estimated_delivery_days), 2) AS avg_estimated_days,
    ROUND(AVG(delay_days), 2) AS avg_delay_days,
    ROUND(SUM(is_on_time) * 100.0 / COUNT(order_id), 2) AS on_time_rate_pct
FROM
    delivery_metrics
GROUP BY
    customer_state
ORDER BY
    total_delivered_orders DESC;
"""

df_delivery = pd.read_sql_query(query_delivery_performance, conn)
display(df_delivery.head(10))

,customer_state,total_delivered_orders,avg_actual_days,avg_estimated_days,avg_delay_days,on_time_rate_pct
0,SP,40494,8.76,19.14,-10.38,94.11
1,RJ,12350,15.31,26.36,-11.05,86.53
2,MG,11354,12.01,24.55,-12.54,94.39
3,RS,5344,15.30,28.51,-13.21,92.85
4,PR,4923,11.99,24.61,-12.62,95.00
5,SC,3546,14.95,25.76,-10.80,90.24
6,BA,3256,19.34,29.43,-10.10,85.96
7,DF,2080,12.97,24.31,-11.34,92.93
8,ES,1995,15.79,25.59,-9.80,87.77
9,GO,1957,15.61,27.09,-11.48,91.82


In [ ]:
query_rfm = """
WITH reference_date AS (
    SELECT MAX(DATE(order_purchase_timestamp)) AS max_date FROM orders
),
customer_aggregations AS (
    SELECT
        c.customer_unique_id,
        MAX(DATE(o.order_purchase_timestamp)) AS last_order_date,
        COUNT(DISTINCT o.order_id) AS frequency,
        ROUND(SUM(p.payment_value), 2) AS monetary
    FROM
        orders o
    JOIN
        customers c ON o.customer_id = c.customer_id
    JOIN
        order_payments p ON o.order_id = p.order_id
    WHERE
        o.order_status = 'delivered'
    GROUP BY
        c.customer_unique_id
),
rfm_raw_scores AS (
    SELECT
        ca.customer_unique_id,
        CAST(JULIANDAY(rd.max_date) - JULIANDAY(ca.last_order_date) AS INTEGER) AS recency_days,
        ca.frequency,
        ca.monetary
    FROM
        customer_aggregations ca
    CROSS JOIN
        reference_date rd
),
rfm_scored AS (
    SELECT
        customer_unique_id,
        recency_days,
        frequency,
        monetary,
        NTILE(4) OVER (ORDER BY recency_days DESC) AS r_score,
        NTILE(4) OVER (ORDER BY monetary ASC) AS m_score
    FROM
        rfm_raw_scores
)
SELECT
    customer_unique_id,
    recency_days,
    frequency,
    monetary,
    r_score,
    m_score,
    CASE
        WHEN r_score = 4 AND m_score = 4 THEN 'Champions'
        WHEN r_score >= 3 AND m_score >= 3 THEN 'Loyal Customers'
        WHEN r_score <= 2 AND m_score >= 3 THEN 'At Risk / Need Attention'
        ELSE 'Low Value / Dormant'
    END AS customer_segment
FROM
    rfm_scored
ORDER BY
    monetary DESC;
"""

df_rfm = pd.read_sql_query(query_rfm, conn)
display(df_rfm.head(10))

,customer_unique_id,recency_days,frequency,monetary,r_score,m_score,customer_segment
0,0a0a92112bd4c708ca5fde585afaa872,383,1,13664.08,2,4,At Risk / Need Attention
1,da122df9eeddfedc1dc1f5349a1a690c,564,2,7571.63,1,4,At Risk / Need Attention
2,763c8b1c9c68a0229c42c9fc6f662b93,94,1,7274.88,4,4,Champions
3,dc4802a71eae9be1dd28f5d788ceb526,612,1,6929.31,1,4,At Risk / Need Attention
4,459bef486812aa25204be022145caa62,84,1,6922.21,4,4,Champions
5,ff4159b92c40ebe40454e3e6a7c35ed6,511,1,6726.66,1,4,At Risk / Need Attention
6,4007669dec559734d6f53e029e360987,327,1,6081.54,2,4,At Risk / Need Attention
7,eebb5dda148d3893cdaf5b5ca3040ccb,547,1,4764.34,1,4,At Risk / Need Attention
8,48e1ac109decbb87765a3eade6854098,117,1,4681.78,4,4,Champions
9,c8460e4251689ba205045f3ea17884a1,70,4,4655.91,4,4,Champions


In [ ]:
query_monthly_growth = """
WITH monthly_metrics AS (
    SELECT
        STRFTIME('%Y-%m', o.order_purchase_timestamp) AS year_month,
        COUNT(DISTINCT o.order_id) AS total_orders,
        ROUND(SUM(p.payment_value), 2) AS total_gmv
    FROM
        orders o
    JOIN
        order_payments p ON o.order_id = p.order_id
    WHERE
        o.order_status = 'delivered'
        AND STRFTIME('%Y-%m', o.order_purchase_timestamp) IS NOT NULL
    GROUP BY
        STRFTIME('%Y-%m', o.order_purchase_timestamp)
)
SELECT
    year_month,
    total_orders,
    total_gmv,
    LAG(total_gmv, 1) OVER (ORDER BY year_month) AS prev_month_gmv,
    ROUND(
        (total_gmv - LAG(total_gmv, 1) OVER (ORDER BY year_month)) * 100.0 /
        LAG(total_gmv, 1) OVER (ORDER BY year_month),
        2
    ) AS mom_gmv_growth_pct
FROM
    monthly_metrics
ORDER BY
    year_month;
"""

df_monthly_growth = pd.read_sql_query(query_monthly_growth, conn)
display(df_monthly_growth.tail(12))

,year_month,total_orders,total_gmv,prev_month_gmv,mom_gmv_growth_pct
10,2017-09,4150,701169.99,646000.61,8.54
11,2017-10,4478,751140.27,701169.99,7.13
12,2017-11,7289,1153528.05,751140.27,53.57
13,2017-12,5513,843199.17,1153528.05,-26.90
14,2018-01,7069,1078606.86,843199.17,27.92
15,2018-02,6555,966510.88,1078606.86,-10.39
16,2018-03,7003,1120678.00,966510.88,15.95
17,2018-04,6798,1132933.95,1120678.00,1.09
18,2018-05,6749,1128836.69,1132933.95,-0.36
19,2018-06,6099,1012090.68,1128836.69,-10.34
